# Testing Explanation Styles with People

This beginner-friendly tutorial helps you plan a study that compares four styles of feature explanation. Participants practice with examples and then answer new questions. The notebook creates group assignments, trial lists, explanation data, and a results table you can fill after collecting responses.

Start with the planning cells. You do not need to understand the model internals to create and inspect the study plan.

## Study design in plain language

A few useful terms:

- **Independent variable (IV):** something you deliberately change, such as the explanation style.
- **Dependent variable (DV):** the outcome you measure, such as the percentage of correct answers.
- **Control variable (CV):** something kept the same so the comparison stays fair.
- **Between-participant design:** different people receive different conditions.
- **Within-participant design:** the same person completes more than one condition.
- **Randomization:** using chance to assign people or question order, which reduces bias.
- **Counterbalancing:** deliberately using different condition orders across people so one order does not dominate the result.

In this tutorial, each person receives one task condition and one explanation style. Everyone completes 10 practice questions and 30 test questions. The main outcome is test accuracy.

In [1]:
from pathlib import Path
import importlib.util
import inspect
import os
import sys
import numpy as np
import pandas as pd

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'api.py').exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Load the dataset class directly so this tutorial does not depend on the legacy `datasets` package alias.
spec = importlib.util.spec_from_file_location('xaikit_tabular_dataset', repo_root / 'assets' / 'original_datasets' / 'tabular_dataset.py')
tabular_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tabular_module)
TabularDataset = tabular_module.TabularDataset

from src.api import xaikitTest
required_trial_args = {'num_training', 'num_testing'}
if not required_trial_args <= set(inspect.signature(xaikitTest.generate_trials).parameters):
    raise RuntimeError('This kernel has an older xaikitTest loaded. Restart the kernel, then Run All.')
from src.ai_models import create_sim2real_function
from src.data_loaders import PreparedDataset
from src.data_loaders.workflow import split_loaded_dataset
from src.workflow_standard import PREDICTION_ONLY_METHOD
from src.xai_adapter import generate_sim2real_explanations

SEED = 42
OUTPUT_DIR = Path(os.environ.get('XAIKIT_TUTORIAL_OUTPUT_DIR', repo_root / 'tutorials' / 'explanation_styles_output'))

In [2]:
condition_map = pd.DataFrame([
    {'validation_condition': 'feature_check', 'task': 'forbidden_features', 'function': 'wiggle', 'memory_budget': 'all_features', 'time_pressure': False},
    {'validation_condition': 'change_prediction', 'task': 'counterfactual_simulation', 'function': 'trend_wiggle', 'memory_budget': 'all_features', 'time_pressure': False},
    {'validation_condition': 'predict_without_timer', 'task': 'forward_simulation', 'function': 'sparse', 'memory_budget': 'all_features', 'time_pressure': False},
    {'validation_condition': 'predict_with_timer', 'task': 'forward_simulation', 'function': 'sparse', 'memory_budget': 'top_2_features', 'time_pressure': True},
])
condition_map

,validation_condition,task,function,memory_budget,time_pressure
0,feature_check,forbidden_features,wiggle,all_features,False
1,change_prediction,counterfactual_simulation,trend_wiggle,all_features,False
2,predict_without_timer,forward_simulation,sparse,all_features,False
3,predict_with_timer,forward_simulation,sparse,top_2_features,True


## 1. Tell the planner what changes and what is measured

The two IVs are `validation_condition` and `explanation_property`. Both are between-participant variables, so each participant is assigned to one combination only. Practice count, test count, wording, and display format are kept constant. The main DV is whether each answer is correct.

Some condition names are custom, so the validator may show support warnings. These warnings do not stop trial planning.

In [3]:
study = xaikitTest(project_name='explanation_style_user_study', output_dir=OUTPUT_DIR)
study.add_iv('validation_condition', 'between', condition_map['validation_condition'].tolist())
study.add_iv('explanation_property', 'between', ['faithful', 'sparse', 'robust', 'sparse_robust'])
study.add_iv('tested_w_xai', 'within', [True, False], randomization='trial')
study.add_cv('num_training', [10])
study.add_cv('num_testing', [30])
study.add_cv('same_instructions', [True])
study.add_cv('same_display_format', [True])
study.add_dv('forward_accuracy', ['continuous'])

pd.DataFrame({'role': ['IV', 'IV', 'IV', 'DV'], 'variable': ['validation_condition', 'explanation_property', 'tested_w_xai', 'forward_accuracy']})

,role,variable
0,IV,validation_condition
1,IV,explanation_property
2,DV,forward_accuracy


## Complete the study setup

Before generating or running questions, write down what the study asks and what participants experience. The code provides editable starting text; replace it with your approved materials. The interactive form lets you revise the same information without editing Python. Saving the form updates `study.study_protocol` and locks any earlier walkthrough approval.


In [4]:
study.set_study_protocol(
    study_title='Understanding different explanation displays',
    research_questions=[
        'RQ1: Does the explanation display change task accuracy?',
        'RQ2: Does the most useful display depend on the task or time limit?',
    ],
    study_summary='Participants practice the task, then answer new questions using one explanation display.',
    consent_text=('Replace this text with your approved consent information. Explain the purpose, '
                  'duration, voluntary participation, risks, data handling, compensation, and contacts.'),
    start_survey_questions=['What is your age range?', 'How familiar are you with AI systems?'],
    end_survey_questions=['How easy was the task?', 'How helpful was the display?'],
    procedure_steps=[
        {'title': 'Welcome and consent', 'kind': 'consent', 'description': 'Read the consent page and choose whether to continue.', 'duration_min': 3},
        {'title': 'Start-of-study survey', 'kind': 'survey', 'description': 'Answer the planned background questions.', 'duration_min': 3},
        {'title': 'Instructions and practice', 'kind': 'practice', 'description': 'Read the instructions and complete 10 practice questions with feedback.', 'duration_min': 7},
        {'title': 'Main questions', 'kind': 'trials', 'description': 'Complete 30 randomized questions without feedback.', 'duration_min': 15},
        {'title': 'End-of-study survey', 'kind': 'survey', 'description': 'Complete the planned ratings.', 'duration_min': 3},
        {'title': 'Debrief', 'kind': 'debrief', 'description': 'Show the debrief and researcher contact information.', 'duration_min': 1},
    ],
)
protocol_editor = study.edit_study_protocol()

## 2. Create questions and participant assignments

The planner needs a pool of example cases. This cell creates a safe synthetic pool, then assigns 10 example participants to every condition combination. Ten is only a quick planning value, not a recommended final sample size. Before recruiting, run a power analysis or consult someone experienced with study statistics.

In [5]:
rng = np.random.default_rng(SEED)
planning_function = create_sim2real_function('sparse')
X = rng.uniform(0, 1, size=(800, planning_function.input_dim))
y = planning_function.predict(X)
dataset = TabularDataset(
    X=X, y=y, feature_names=list(planning_function.feature_names),
    categorical_feature_options={}, ordinal_feature_indices=[],
    target_name='alien_health', target_options=['healthy', 'sick'], dataset_name='sim2real',
)
study.data = PreparedDataset(split_loaded_dataset('sim2real', dataset, one_hot_encode=False, test_size=0.5, random_state=SEED))

trial_result = study.generate_trials(
    participants_per_between_condition=10, num_training=10, num_testing=30,
    trial_randomization_strategy='random', shuffle_instances=True,
    output_dir='trials', seed=SEED, preview_rows=4,
)
trials = pd.DataFrame(trial_result.trials).merge(condition_map, on='validation_condition', how='left')
trials['xai_method'] = trials['function'] + '__' + trials['explanation_property']
assert (trials.groupby('participantId').head(10)['phase'] == 'training').all()
assert trials.query("phase == 'training'")['tested_w_xai'].isna().all()
assert trials.query("phase == 'testing'")['tested_w_xai'].notna().all()
trials.groupby(['validation_condition', 'explanation_property']).agg(participants=('participantId', 'nunique'), trials=('trialId', 'count')).head(8)

Counterbalancing strategy: complete_counterbalancing
Participant assignments: 160 total
Instance pool rows: 300
Trial rows: 6400
Exported trial artifacts:
  CSV     : /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/tutorials/explanation_styles_output/trials/trials.csv
  JSON    : /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/tutorials/explanation_styles_output/trials/trials.json
  Summary : /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/tutorials/explanation_styles_output/trials/design_summary.json

Previewing first 4 trial rows:
{'participantId': 1, 'trialId': 1, 'block': 0, 'trialWithinBlock': 1, 'withinCondition': 'training', 'validation_condition': 'feature_check', 'explanation_property': 'faithful', 'dataId': 'sim2real', 'instanceId': '435'}
{'participantId': 1, 'trialId': 2, 'block': 0, 'trialWithinBlock': 2, 'withinCondition': 'training', 'validation_condition': 'feature_check', 'explanation_property': 'faithful', 'dataId': 'sim2real', 'instanceId': '5

participants  trials
validation_condition explanation_property                      
change_prediction    faithful                        10     400
                     robust                          10     400
                     sparse                          10     400
                     sparse_robust                   10     400
feature_check        faithful                        10     400
                     robust                          10     400
                     sparse                          10     400
                     sparse_robust                   10     400

## 3. Generate the four explanation styles

The styles represent different trade-offs:

- `faithful`: aims to closely reflect the model.
- `sparse`: shows fewer features.
- `robust`: changes less for similar cases.
- `sparse_robust`: combines fewer features with greater stability.

The audit table helps verify that the generated explanations differ in the intended way before any participant sees them.

In [6]:
properties = ['faithful', 'sparse', 'robust', 'sparse_robust']
explanation_audit = []
explanation_results = {}
for function_name in condition_map['function'].unique():
    function = create_sim2real_function(function_name)
    X_demo = rng.uniform(0, 1, size=(30, function.input_dim))
    generated = generate_sim2real_explanations(
        model=function, X=X_demo, properties=properties, top_k=2,
        n_candidate_explanations=250, n_local_samples=100, random_state=SEED,
    )
    explanation_results[function_name] = generated
    for prop, result in generated.items():
        explanation_audit.append({'function': function_name, 'property': prop, 'strategy': result.metadata['strategy'], **result.metadata['property_metrics']})

audit_df = pd.json_normalize(explanation_audit)
audit_df

,function,property,strategy,faithfulness_loss,sparsity_loss,robustness_loss
0,wiggle,faithful,ground_truth_weights,0.000000,9.433333,2.480766
1,wiggle,sparse,top_k_from_faithful_confound_adjusted,0.266667,2.000000,5.366270
2,wiggle,robust,sample_global_explanations,0.433333,11.000000,0.000000
3,wiggle,sparse_robust,sample_global_explanations_only_top_k,0.433333,2.000000,0.000000
4,trend_wiggle,faithful,fit_locally_faithful_line,11.001184,7.000000,342.048540
5,trend_wiggle,sparse,top_k_from_faithful_confound_adjusted,25072.196719,2.000000,806.963318
6,trend_wiggle,robust,ground_truth_trend_weights,74.480479,4.000000,0.000000
7,trend_wiggle,sparse_robust,sample_global_explanations_only_top_k,125.703046,2.000000,0.000000
8,sparse,faithful,ground_truth_weights_confound_adjusted,0.233333,3.000000,6.541364
9,sparse,sparse,ground_truth_weights,0.000000,1.000000,4.088353


## 4. Use a consistent participant procedure

For each participant:

1. Show the consent form and study instructions.
2. Ask a few simple questions to confirm the instructions are understood.
3. Give 10 practice questions with the correct answer shown.
4. Give 30 test questions in random order without feedback.
5. Show a short exit survey.

Use the timer only in the timed condition. Decide exclusion rules before collecting data, keep a log of exclusions, and obtain ethics approval where required.

In [7]:
human_response_columns = [
    'participantId', 'validation_condition', 'explanation_property', 'trialId',
    'instanceId', 'response', 'correct_response', 'task_correct', 'response_time_s',
    'comprehension_pass', 'ease_of_use_likert', 'perceived_sparsity_likert',
    'perceived_robustness_likert', 'perceived_faithfulness_likert', 'helpfulness_likert',
]
pd.DataFrame(columns=human_response_columns)

,participantId,validation_condition,explanation_property,trialId,instanceId,response,correct_response,task_correct,response_time_s,comprehension_pass,ease_of_use_likert,perceived_sparsity_likert,perceived_robustness_likert,perceived_faithfulness_likert,helpfulness_likert


## 5. Summarize the results

First calculate one accuracy value per participant, then summarize those values by condition and explanation style. Do not compare different tasks as though they were the same outcome. The simple table below is a good quality check, but it is not enough for a publishable claim. Choose formal statistical tests before data collection with help from a qualified analyst.

In [8]:
def summarize_participant_accuracy(responses):
    clean = responses.query('comprehension_pass == True').copy()
    participant_accuracy = clean.groupby(
        ['participantId', 'validation_condition', 'explanation_property'], as_index=False
    )['task_correct'].mean()
    return participant_accuracy.groupby(
        ['validation_condition', 'explanation_property']
    )['task_correct'].agg(['count', 'mean', 'std']).reset_index()

print('After importing responses, call summarize_participant_accuracy(human_responses).')

After importing responses, call summarize_participant_accuracy(human_responses).


## Replication checks and expected result pattern

The reference setup uses four between-participant task conditions crossed with four explanation styles, 10 practice questions, and 30 randomized test questions. The timed prediction condition displays a global and per-question timer. Use the same selected cases within a condition so explanation style is the main difference.

Expected descriptive patterns are shown below. Treat them as replication targets, not guaranteed outcomes. Report confidence intervals and departures from the target pattern rather than hiding them.

In [9]:
expected_patterns = pd.DataFrame([
    {'condition': 'feature_check', 'expected_pattern': 'faithful explanations have the highest accuracy'},
    {'condition': 'change_prediction', 'expected_pattern': 'robust explanations have the highest accuracy'},
    {'condition': 'predict_without_timer', 'expected_pattern': 'faithful and sparse explanations perform similarly'},
    {'condition': 'predict_with_timer', 'expected_pattern': 'sparse explanations have the highest accuracy'},
])
expected_patterns

,condition,expected_pattern
0,feature_check,faithful explanations have the highest accuracy
1,change_prediction,robust explanations have the highest accuracy
2,predict_without_timer,faithful and sparse explanations perform simil...
3,predict_with_timer,sparse explanations have the highest accuracy


## 6. Run the complete study with virtual participants

This cell uses the KNN baseline from the cognitive-model API. The executor fits a fresh KNN on each participant-condition training phase, then records its responses on that condition's held-out testing phase. These are machine-proxy responses, not human evidence.

The `prediction_pool` is the AI answer key required by the executor. The results are saved as CSV and JSON, then averaged by experimental condition.

In [10]:
study.trials = trials.to_dict('records')
study.trained_ai_model = planning_function
study.model_name = 'tutorial_function'

explanation_frames = []
for (function_name, property_name, method_name), group in trials.groupby(
    ['function', 'explanation_property', 'xai_method'], sort=False
):
    trial_ids = sorted(group['instanceId'].astype(int).unique())
    X_trials = study.data.split.X_model[trial_ids]
    function = create_sim2real_function(function_name)
    result = generate_sim2real_explanations(
        model=function, X=X_trials, properties=[property_name], top_k=2,
        n_candidate_explanations=250, n_local_samples=100, random_state=SEED,
    )[property_name]
    frame = result.to_explanation_df(
        instance_ids=trial_ids, predictions=function.predict(X_trials),
        dataset_id=study.data.dataset_id, model_name=study.model_name,
    )
    frame['expMethod'] = method_name
    explanation_frames.append(frame)
prediction_pool = pd.concat(explanation_frames, ignore_index=True)
study.set_cognitive_model(
    cognitive_model_id='knn',
    model_kwargs={'n_neighbors': 3},
)

## 7. Preview and approve the complete participant journey

Run this cell before execution. Use **Back** and **Next** to inspect the researcher summary, consent page, start survey, practice instructions, every question for one participant, end survey, and debrief. On the last page, tick the confirmation box and click **Approve walkthrough**. Any later edit to the setup requires a new approval.


In [ ]:
walkthrough_pages = study.preview_experiment_walkthrough(
    participant_id=30, explanation_pool=prediction_pool, visualization='importance',
)

## 8. Execute only after approval

The API refuses to run this cell until the walkthrough has been approved. This prevents a researcher from accidentally simulating an unreviewed setup.


In [12]:
protocol_path = study.save_study_protocol()
simulated_results = study.run_experiment(
    mode='whole_experiment', participant_id=None, explanation_pool=prediction_pool,
    require_walkthrough_approval=True,
)
csv_path, json_path = study.save_results(out_dir='simulated_results')
simulation_summary = simulated_results.query('phase == "testing"').groupby(
    ['validation_condition', 'explanation_property'], as_index=False
).agg(participants=('participantId', 'nunique'), questions=('trialId', 'size'), accuracy=('forward_accuracy', 'mean'))
print(f'Saved the setup to {protocol_path}')
print(f'Saved {len(simulated_results):,} virtual responses to {csv_path} and {json_path}')
iv_dv_analysis = study.analyze_iv_dv(iv='explanation_property', dv='forward_accuracy')
display(simulation_summary, iv_dv_analysis.descriptives)

RuntimeError: Experiment execution is locked. Preview the complete walkthrough and click 'Approve walkthrough' first.

## Before recruiting anyone

- Replace synthetic cases with the finalized replication stimuli.
- Pilot the instructions with people who are not on the project.
- Confirm that each condition has the intended number and difficulty of questions.
- Predefine exclusions, outcomes, sample size, and analysis.
- Confirm consent, privacy, compensation, and ethics requirements.
- Save the random seed and final trial files so the study can be audited.
- Keep virtual and human results clearly labeled and stored separately.